# Patient fact extraction: Jev vs Claude

This notebook compares TypeSafe Jev with Claude on extracting severity, age, and medication names from patient messages. It only extracts stated facts; it does not give medical advice, a diagnosis, or a risk level.

**Jev limitation:** Jev returns typed decisions over configured answer choices, rather than generating arbitrary text. Severity is a fixed set of labels, age is limited to 0–130 years, and medications are detected only from the editable vocabulary below. Claude gets the full open-ended medication extraction task. A medication omitted from Jev's vocabulary cannot be returned, so this is not an apples-to-apples comparison for unknown medication names.

Set `TYPESAFE_API_KEY`, `TYPESAFE_BASE_URL`, and `ANTHROPIC_API_KEY` in this folder's `.env` before running. The sample messages are synthetic.

In [1]:
# Install dependencies if needed:
# %pip install python-dotenv typesafe_sdk anthropic pydantic
import os
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, field_validator
from typesafe_sdk import Choice, Noul, TypeSafeClient

load_dotenv()

required_typesafe_vars = ("TYPESAFE_API_KEY", "TYPESAFE_BASE_URL")
missing_typesafe_vars = [name for name in required_typesafe_vars if not os.getenv(name)]
if missing_typesafe_vars:
    raise RuntimeError(f"Set these environment variables before running: {', '.join(missing_typesafe_vars)}")

MODEL_NAME = "jev-latest"
jev_client = TypeSafeClient(
    api_key=os.environ["TYPESAFE_API_KEY"],
    base_url=os.environ["TYPESAFE_BASE_URL"],
)

In [2]:
class PatientFacts(BaseModel):
    model_config = ConfigDict(extra="forbid")

    severity: int | None = Field(ge=1, le=10, strict=True)
    age_years: int | None = Field(ge=0, strict=True)
    medications: list[str]

    @field_validator("medications")
    @classmethod
    def normalize_medications(cls, values: list[str]) -> list[str]:
        return sorted({value.strip().lower() for value in values if value.strip()})


# Jev's medication detector is intentionally bounded to this editable vocabulary.
MEDICATION_VOCABULARY = [
    "acetaminophen",
    "albuterol",
    "amoxicillin",
    "apixaban",
    "aspirin",
    "atorvastatin",
    "ibuprofen",
    "lisinopril",
    "metformin",
]

SEVERITY_CHOICES = {
    "NOT_STATED": "No numeric pain or discomfort level is stated.",
    **{str(value): f"The patient explicitly gives a pain or discomfort level of {value} out of 10." for value in range(1, 11)},
}
AGE_CHOICES = {
    "NOT_STATED": "The patient does not explicitly state a whole-number age in years.",
    **{str(value): f"The patient explicitly states they are {value} years old." for value in range(0, 131)},
}

def extract_with_jev(message: str) -> dict:
    medication_question_names = {
        f"takes_med_{index}": medication
        for index, medication in enumerate(MEDICATION_VOCABULARY)
    }
    questions = {
        "severity": Choice(
            instructions=(
                "Select the explicitly stated numeric pain or discomfort level from 1 to 10. "
                "Choose NOT_STATED if the patient gives no number; do not infer a level from words."
            ),
            criteria=SEVERITY_CHOICES,
        ),
        "age_years": Choice(
            instructions=(
                "Select the patient's explicitly stated age in whole years. "
                "Choose NOT_STATED if no age in years is given; do not infer or convert an age."
            ),
            criteria=AGE_CHOICES,
        ),
    }
    for question_name, medication in medication_question_names.items():
        questions[question_name] = Noul(
            instructions=(
                f"The patient explicitly says they take or use {medication}. "
                "Do not count it if it is only an allergy, a question, a negation, or mentioned as someone else's medication."
            )
        )
    print(questions)
    response = jev_client.system_one(
        model=MODEL_NAME,
        state={"patient_message": message},
        questions=questions,
    )
    severity_choice = response.answers["severity"].choice
    age_choice = response.answers["age_years"].choice
    medications = [
        medication
        for question_name, medication in medication_question_names.items()
        if response.answers[question_name].noul >= 0.5
    ]

    return PatientFacts(
        severity=None if severity_choice == "NOT_STATED" else int(severity_choice),
        age_years=None if age_choice == "NOT_STATED" else int(age_choice),
        medications=medications,
    ).model_dump()


messages = [
    "I'm 42, my pain is about a 7, and I take metformin and lisinopril.",
    "I'm 8 years old and my stomach hurts badly. I use albuterol when it flares.",
    "I'm 36. The discomfort is a 3 out of 10. I took sertraline this morning.",
    "Pain is 9/10. I take aspirin. I'm 67.",
]

jev_results = [extract_with_jev(message) for message in messages]
for message, facts in zip(messages, jev_results):
    print({"message": message, "jev": facts})

{'severity': Choice(type='choice', instructions='Select the explicitly stated numeric pain or discomfort level from 1 to 10. Choose NOT_STATED if the patient gives no number; do not infer a level from words.', criteria={'NOT_STATED': 'No numeric pain or discomfort level is stated.', '1': 'The patient explicitly gives a pain or discomfort level of 1 out of 10.', '2': 'The patient explicitly gives a pain or discomfort level of 2 out of 10.', '3': 'The patient explicitly gives a pain or discomfort level of 3 out of 10.', '4': 'The patient explicitly gives a pain or discomfort level of 4 out of 10.', '5': 'The patient explicitly gives a pain or discomfort level of 5 out of 10.', '6': 'The patient explicitly gives a pain or discomfort level of 6 out of 10.', '7': 'The patient explicitly gives a pain or discomfort level of 7 out of 10.', '8': 'The patient explicitly gives a pain or discomfort level of 8 out of 10.', '9': 'The patient explicitly gives a pain or discomfort level of 9 out of 10

In [4]:
import json
import anthropic

CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in the .env file before running the Claude comparison cell.")

claude_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
EXTRACTION_PROMPT = """You extract three facts from a patient's message for a triage assistant.
severity: the pain or discomfort level the patient states, as a whole number from 1 to 10
(for example "about a 7" gives 7). If they give no number, use null; never guess one.
age_years: the patient's age in years if stated, otherwise null.
medications: medications the patient says they take, as short lowercase names; empty if none.
Reply only with JSON matching the schema. You never give medical advice, a diagnosis, or a risk
level; you only extract.

The JSON object must contain exactly these keys: severity, age_years, medications. Use integer
values or null for severity and age_years, and an array of lowercase strings for medications."""

def extract_with_claude(message: str) -> dict:
    response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=256,
        system=EXTRACTION_PROMPT,
        messages=[{"role": "user", "content": message}],
    )
    text = "".join(block.text for block in response.content if block.type == "text")
    payload = json.loads(text)
    return PatientFacts.model_validate(payload).model_dump()


claude_results = [extract_with_claude(message) for message in messages]
for message, facts in zip(messages, claude_results):
    print({"message": message, "claude": facts})

{'message': "I'm 42, my pain is about a 7, and I take metformin and lisinopril.", 'claude': {'severity': 7, 'age_years': 42, 'medications': ['lisinopril', 'metformin']}}
{'message': "I'm 8 years old and my stomach hurts badly. I use albuterol when it flares.", 'claude': {'severity': None, 'age_years': 8, 'medications': ['albuterol']}}
{'message': "I'm 36. The discomfort is a 3 out of 10. I took sertraline this morning.", 'claude': {'severity': 3, 'age_years': 36, 'medications': ['sertraline']}}
{'message': "Pain is 9/10. I take aspirin. I'm 67.", 'claude': {'severity': 9, 'age_years': 67, 'medications': ['aspirin']}}


In [5]:
FIELDS = ("severity", "age_years", "medications")
comparison = []
for message, jev_facts, claude_facts in zip(messages, jev_results, claude_results):
    field_matches = {
        field: jev_facts[field] == claude_facts[field]
        for field in FIELDS
    }
    comparison.append({
        "message": message,
        "jev": jev_facts,
        "claude": claude_facts,
        "field_matches": field_matches,
        "all_fields_match": all(field_matches.values()),
    })

for row in comparison:
    print(json.dumps(row, indent=2, ensure_ascii=False))

field_accuracy = {
    field: sum(row["field_matches"][field] for row in comparison) / len(comparison)
    for field in FIELDS
}
overall_exact_match = sum(row["all_fields_match"] for row in comparison) / len(comparison)
print("Per-field exact match rate:", field_accuracy)
print("All-fields exact match rate:", overall_exact_match)
print("Note: Jev cannot detect sertraline in the third sample because it is not in MEDICATION_VOCABULARY.")

{
  "message": "I'm 42, my pain is about a 7, and I take metformin and lisinopril.",
  "jev": {
    "severity": 7,
    "age_years": 42,
    "medications": [
      "lisinopril",
      "metformin"
    ]
  },
  "claude": {
    "severity": 7,
    "age_years": 42,
    "medications": [
      "lisinopril",
      "metformin"
    ]
  },
  "field_matches": {
    "severity": true,
    "age_years": true,
    "medications": true
  },
  "all_fields_match": true
}
{
  "message": "I'm 8 years old and my stomach hurts badly. I use albuterol when it flares.",
  "jev": {
    "severity": null,
    "age_years": 8,
    "medications": [
      "albuterol"
    ]
  },
  "claude": {
    "severity": null,
    "age_years": 8,
    "medications": [
      "albuterol"
    ]
  },
  "field_matches": {
    "severity": true,
    "age_years": true,
    "medications": true
  },
  "all_fields_match": true
}
{
  "message": "I'm 36. The discomfort is a 3 out of 10. I took sertraline this morning.",
  "jev": {
    "severity": 3

In [6]:
import pandas as pd

rows = []
for message, jev_facts, claude_facts in zip(messages, jev_results, claude_results):
    jev_med = ", ".join(jev_facts.get("medications", [])) if jev_facts.get("medications") else "—"
    claude_med = ", ".join(claude_facts.get("medications", [])) if claude_facts.get("medications") else "—"

    field_matches = {
        "severity": jev_facts.get("severity") == claude_facts.get("severity"),
        "age_years": jev_facts.get("age_years") == claude_facts.get("age_years"),
        "medications": jev_facts.get("medications") == claude_facts.get("medications"),
    }

    rows.append({
        "message": message,
        "Jev severity": jev_facts.get("severity"),
        "Claude severity": claude_facts.get("severity"),
        "Jev age": jev_facts.get("age_years"),
        "Claude age": claude_facts.get("age_years"),
        "Jev medications": jev_med,
        "Claude medications": claude_med,
        "Severity match": field_matches["severity"],
        "Age match": field_matches["age_years"],
        "Medications match": field_matches["medications"],
        "All fields match": all(field_matches.values()),
    })

comparison_df = pd.DataFrame(rows)
comparison_df

,message,Jev severity,Claude severity,Jev age,Claude age,Jev medications,Claude medications,Severity match,Age match,Medications match,All fields match
0,"I'm 42, my pain is about a 7, and I take metfo...",7.0,7.0,42,42,"lisinopril, metformin","lisinopril, metformin",True,True,True,True
1,I'm 8 years old and my stomach hurts badly. I ...,NaN,NaN,8,8,albuterol,albuterol,True,True,True,True
2,I'm 36. The discomfort is a 3 out of 10. I too...,3.0,3.0,36,36,—,sertraline,True,True,False,False
3,Pain is 9/10. I take aspirin. I'm 67.,9.0,9.0,67,67,aspirin,aspirin,True,True,True,True
